# Multi-task Nano-QSAR model project: hyperparameters tuning with BayesSearchCV
Author:Kun Mi

Description: In this study, we multi-task DNN model to predict Nanoparticles biodistribution and delivery efficiency in liver, kideny, spleen, lung and tumor.



# Install and import required python pacakges

In [ ]:
!pip install -q shap scikit-optimize
! pip install -q optuna
! pip install -q tensorflow
!pip install -q keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 3.2 MB/s eta 0:00:00


In [ ]:
#Install basic python pcakges
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import os
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import mean_squared_log_error
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from time import time

# Link to google drive and set up your project folder

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/MTL/Multi_output_regression/Data_parameter_based.csv')

Mounted at /content/drive


In [ ]:
target_cols = ['DE_tumor', 'DE_heart', 'DE_liver', 'DE_spleen', 'DE_lung', 'DE_kidney']
Y_re = df[target_cols]
X_re = df.drop(columns=target_cols)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test= train_test_split(
    X_re, Y_re,
    test_size=0.2,
    random_state=3407
)

print(X_train.shape, X_test.shape)

(238, 28) (60, 28)


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

cols_num = ['Size', 'Zeta Potential']

mscaler = MinMaxScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[cols_num] = mscaler.fit_transform(X_train[cols_num])
X_test_scaled[cols_num] = mscaler.transform(X_test[cols_num])

print(X_train_scaled.head())
print(X_test_scaled.head())

     MAT_Dendrimer  MAT_Gold  MAT_Hybrid  MAT_Hydrogel  MAT_Iron Oxide  \
10             0.0       1.0         0.0           0.0             0.0   
90             0.0       0.0         0.0           1.0             0.0   
239            0.0       0.0         0.0           0.0             0.0   
99             0.0       0.0         0.0           0.0             0.0   
73             0.0       0.0         0.0           0.0             0.0   

     MAT_Liposome  MAT_Other IM  MAT_Other OM  MAT_Polymeric  MAT_Silica  ...  \
10            0.0           0.0           0.0            0.0         0.0  ...   
90            0.0           0.0           0.0            0.0         0.0  ...   
239           0.0           1.0           0.0            0.0         0.0  ...   
99            0.0           0.0           0.0            1.0         0.0  ...   
73            0.0           0.0           0.0            1.0         0.0  ...   

     TM_Allograft Orthotopic  TM_Xenograft Heterotopic  \
10        

In [ ]:
#Name X_train and X_test
X_train= X_train_scaled.to_numpy()
X_test= X_test_scaled.to_numpy()

# Traditional ML

## MultiOutputRegress

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error

def evaluate_multioutput_model(y_true_df, y_pred_df, digits=2):
    results = []

    # Check if y_pred_df is not a DataFrame, and convert if necessary
    if not isinstance(y_pred_df, pd.DataFrame):
        y_pred_df = pd.DataFrame(y_pred_df, columns=y_true_df.columns, index=y_true_df.index)

    for target_col in y_true_df.columns:
        y_true_col = y_true_df[target_col]
        y_pred_col = y_pred_df[target_col]

        rmse = np.sqrt(mean_squared_error(y_true_col, y_pred_col))
        r2 = r2_score(y_true_col, y_pred_col)

        results.append({
            'Target': target_col,
            'RMSE': rmse,
            'R2': r2
        })

    results_df = pd.DataFrame(results)
    return results_df.round(digits)

## KRR

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import root_mean_squared_error as rmse
from sklearn.kernel_ridge import KernelRidge
import optuna
import numpy as np

def objective(trial):

    params = {
        "alpha": trial.suggest_float("alpha", 1e-5, 10, log=True),
        "kernel": trial.suggest_categorical("kernel", ["rbf", "poly", "sigmoid"]),
        "gamma": trial.suggest_float("gamma", 1e-5, 1, log=True),
        "degree": trial.suggest_int("degree", 2, 4),
    }

    model = KernelRidge(**params)

    rmse_scores = []

    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    rmse_scores = []
    for tr_idx, va_idx in cv.split(X_train):
        X_tr, X_va = X_train[tr_idx], X_train[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_va)
        rmse_scores.append(rmse(y_va, y_pred))

    return np.mean(rmse_scores)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

print("Best params:", study.best_params)

[I 2026-06-12 13:01:49,339] A new study created in memory with name: no-name-1590a5db-c5e2-4866-ba89-f9f9443a7df0
[I 2026-06-12 13:01:49,543] Trial 0 finished with value: 1.6091392176406534 and parameters: {'alpha': 5.720204469741581e-05, 'kernel': 'rbf', 'gamma': 0.0001129772946516078, 'degree': 4}. Best is trial 0 with value: 1.6091392176406534.
[I 2026-06-12 13:01:49,633] Trial 1 finished with value: 1.6497009837190124 and parameters: {'alpha': 1.091651046257166, 'kernel': 'sigmoid', 'gamma': 1.1764833359659136e-05, 'degree': 3}. Best is trial 0 with value: 1.6091392176406534.
[I 2026-06-12 13:01:49,693] Trial 2 finished with value: 1.630590369706504 and parameters: {'alpha': 0.0509288073204127, 'kernel': 'rbf', 'gamma': 9.067652629180474e-05, 'degree': 4}. Best is trial 0 with value: 1.6091392176406534.
[I 2026-06-12 13:01:49,848] Trial 3 finished with value: 1.6027014804847775 and parameters: {'alpha': 0.006845801537262906, 'kernel': 'rbf', 'gamma': 5.2811597395119355e-05, 'degree

Best params: {'alpha': 1.2526471996995485, 'kernel': 'poly', 'gamma': 0.07185862704754453, 'degree': 4}


In [ ]:
#Evaluation for test set
best_params = study.best_params
Best_model= KernelRidge(**best_params)

# Fit the best model to the training data
Best_model.fit(X_train, y_train)
#Evaluation for the test data
y_pred_test= Best_model.predict(X_test)
y_pred_test_df = pd.DataFrame(y_pred_test, columns=target_cols, index=y_test.index)
test_metrics = evaluate_multioutput_model(y_test, y_pred_test_df)

print("\n--- Test Set Evaluation Results (per tissue) ---")
print(test_metrics)


--- Test Set Evaluation Results (per tissue) ---
      Target  RMSE    R2
0   DE_tumor  1.77  0.02
1   DE_heart  1.65  0.03
2   DE_liver  1.60  0.06
3  DE_spleen  1.77 -0.07
4    DE_lung  1.46  0.02
5  DE_kidney  1.42  0.03


In [ ]:
from sklearn.model_selection import KFold
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor

# Initialize KFold for cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# List to store evaluation metrics for each fold
cv_results = []

# Loop through each fold
for fold, (train_index, val_index) in enumerate(kf.split(X_train), 1):

    # Split data for the current fold
    X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]
    y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

    # Build and train the model for this fold using the best parameters
    model = KernelRidge(**best_params)
    model.fit(X_train_fold, y_train_fold)

    # Make predictions on the validation set
    y_val_pred = model.predict(X_val_fold)

    # Evaluate per-tissue metrics for this fold
    fold_metrics_df = evaluate_multioutput_model(y_val_fold, y_val_pred)
    fold_metrics_df['Fold'] = fold
    cv_results.append(fold_metrics_df)

# Concatenate all fold results and calculate mean and std deviation
all_cv_metrics = pd.concat(cv_results)
cv_summary = all_cv_metrics.groupby('Target').agg(
    RMSE_mean=('RMSE', 'mean'),
    RMSE_std=('RMSE', 'std'),
    R2_mean=('R2', 'mean'),
    R2_std=('R2', 'std')
).round(3)
print(cv_summary)

           RMSE_mean  RMSE_std  R2_mean  R2_std
Target                                         
DE_heart       1.648     0.135   -0.068   0.167
DE_kidney      1.652     0.163   -0.044   0.292
DE_liver       1.390     0.164    0.144   0.253
DE_lung        1.482     0.224    0.012   0.198
DE_spleen      1.634     0.211    0.162   0.216
DE_tumor       1.480     0.097    0.010   0.222


## Random forest

In [ ]:
import optuna
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error as rmse
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "max_depth": trial.suggest_int("max_depth", 3, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
    }

    model =  RandomForestRegressor(
            **params,
            random_state=42,
            n_jobs=-1

    )
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    rmse_scores = []
    for tr_idx, va_idx in cv.split(X_train):
        X_tr, X_va = X_train[tr_idx], X_train[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_va)
        rmse_scores.append(rmse(y_va, y_pred))

    return np.mean(rmse_scores)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

print("Best params:", study.best_params)

[I 2026-06-12 13:02:10,123] A new study created in memory with name: no-name-35554015-49ab-464c-8d4c-802e33656484
[I 2026-06-12 13:02:13,721] Trial 0 finished with value: 1.5190566913379642 and parameters: {'n_estimators': 178, 'max_depth': 27, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': None}. Best is trial 0 with value: 1.5190566913379642.
[I 2026-06-12 13:02:18,179] Trial 1 finished with value: 1.5571374415506698 and parameters: {'n_estimators': 410, 'max_depth': 5, 'min_samples_split': 4, 'min_samples_leaf': 4, 'max_features': 'log2'}. Best is trial 0 with value: 1.5190566913379642.
[I 2026-06-12 13:02:19,899] Trial 2 finished with value: 1.5519068783067218 and parameters: {'n_estimators': 149, 'max_depth': 29, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': None}. Best is trial 0 with value: 1.5190566913379642.
[I 2026-06-12 13:02:25,505] Trial 3 finished with value: 1.5411685763611702 and parameters: {'n_estimators': 441, 'max_depth': 7, 'min_sam

Best params: {'n_estimators': 218, 'max_depth': 28, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt'}


In [ ]:
#Evaluation for test set
best_params = study.best_params
Best_model= RandomForestRegressor(**best_params,random_state=42,n_jobs=-1)

# Fit the best model to the training data
Best_model.fit(X_train, y_train)
#Evaluation for the test data
y_pred_test= Best_model.predict(X_test)
y_pred_test_df = pd.DataFrame(y_pred_test, columns=target_cols, index=y_test.index)
test_metrics = evaluate_multioutput_model(y_test, y_pred_test_df)

print("\n--- Test Set Evaluation Results (per tissue) ---")
print(test_metrics)


--- Test Set Evaluation Results (per tissue) ---
      Target  RMSE    R2
0   DE_tumor  1.74  0.05
1   DE_heart  1.33  0.37
2   DE_liver  1.45  0.23
3  DE_spleen  1.62  0.10
4    DE_lung  1.29  0.24
5  DE_kidney  1.30  0.18


In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor

# Initialize KFold for cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# List to store evaluation metrics for each fold
cv_results = []

# Loop through each fold
for fold, (train_index, val_index) in enumerate(kf.split(X_train), 1):

    # Split data for the current fold
    X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]
    y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

    # Build and train the model for this fold using the best parameters
    model = RandomForestRegressor(**best_params,random_state=42,n_jobs=-1)
    model.fit(X_train_fold, y_train_fold)

    # Make predictions on the validation set
    y_val_pred = model.predict(X_val_fold)

    # Evaluate per-tissue metrics for this fold
    fold_metrics_df = evaluate_multioutput_model(y_val_fold, y_val_pred)
    fold_metrics_df['Fold'] = fold
    cv_results.append(fold_metrics_df)

# Concatenate all fold results and calculate mean and std deviation
all_cv_metrics = pd.concat(cv_results)
cv_summary = all_cv_metrics.groupby('Target').agg(
    RMSE_mean=('RMSE', 'mean'),
    RMSE_std=('RMSE', 'std'),
    R2_mean=('R2', 'mean'),
    R2_std=('R2', 'std')
).round(3)
print(cv_summary)

           RMSE_mean  RMSE_std  R2_mean  R2_std
Target                                         
DE_heart       1.494     0.125    0.108   0.209
DE_kidney      1.428     0.079    0.192   0.300
DE_liver       1.292     0.112    0.264   0.191
DE_lung        1.330     0.161    0.190   0.190
DE_spleen      1.510     0.145    0.278   0.176
DE_tumor       1.394     0.098    0.124   0.186


## SVR

In [ ]:
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import KFold
import optuna
import numpy as np

def objective(trial):

    params = {
        "C": trial.suggest_float("C", 1e-2, 100, log=True),
        "epsilon": trial.suggest_float("epsilon", 1e-3, 1.0, log=True),
        "gamma": trial.suggest_float("gamma", 1e-4, 1, log=True)
    }

    model = MultiOutputRegressor(SVR(**params))

    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    rmse_scores = []

    for tr_idx, va_idx in cv.split(X_train):
        X_tr, X_va = X_train[tr_idx], X_train[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_va)
        rmse_scores.append(rmse(y_va, y_pred))

    return np.mean(rmse_scores)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

print("Best params:", study.best_params)

[I 2026-06-12 13:05:43,109] A new study created in memory with name: no-name-f483b7a2-f329-4d17-a2ae-37d453e20a2a
[I 2026-06-12 13:05:43,341] Trial 0 finished with value: 1.6274921367876687 and parameters: {'C': 28.929785487637186, 'epsilon': 0.023905606140794962, 'gamma': 0.03135103775891956}. Best is trial 0 with value: 1.6274921367876687.
[I 2026-06-12 13:05:43,511] Trial 1 finished with value: 1.5923515988449428 and parameters: {'C': 0.5311702635898798, 'epsilon': 0.020585857848156263, 'gamma': 0.1585627991811409}. Best is trial 1 with value: 1.5923515988449428.
[I 2026-06-12 13:05:43,701] Trial 2 finished with value: 1.6233935144187792 and parameters: {'C': 50.12863849800087, 'epsilon': 0.015997198267782484, 'gamma': 0.004002545329907561}. Best is trial 1 with value: 1.5923515988449428.
[I 2026-06-12 13:05:43,867] Trial 3 finished with value: 1.6629584282092675 and parameters: {'C': 0.02067589412656194, 'epsilon': 0.04297242049206997, 'gamma': 0.014637868256779998}. Best is trial 

Best params: {'C': 1.6200773322032385, 'epsilon': 0.9545361366288293, 'gamma': 0.974157068190987}


In [ ]:
#Evaluation for test set
best_params= study.best_params
Best_model=MultiOutputRegressor(SVR(**best_params))

# Fit the best model to the training data
Best_model.fit(X_train, y_train)
#Evaluation for the test data
y_pred_test= Best_model.predict(X_test)
y_pred_test_df = pd.DataFrame(y_pred_test, columns=target_cols, index=y_test.index)
test_metrics = evaluate_multioutput_model(y_test, y_pred_test_df)

print("\n--- Test Set Evaluation Results (per tissue) ---")
print(test_metrics)


--- Test Set Evaluation Results (per tissue) ---
      Target  RMSE    R2
0   DE_tumor  1.77  0.02
1   DE_heart  1.64  0.04
2   DE_liver  1.56  0.10
3  DE_spleen  1.75 -0.05
4    DE_lung  1.46  0.02
5  DE_kidney  1.37  0.10


In [ ]:
from sklearn.model_selection import KFold
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor

# Initialize KFold for cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# List to store evaluation metrics for each fold
cv_results = []

# Loop through each fold
for fold, (train_index, val_index) in enumerate(kf.split(X_train), 1):

    # Split data for the current fold
    X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]
    y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

    # Build and train the model for this fold using the best parameters
    model = MultiOutputRegressor(SVR(**best_params))
    model.fit(X_train_fold, y_train_fold)

    # Make predictions on the validation set
    y_val_pred = model.predict(X_val_fold)

    # Evaluate per-tissue metrics for this fold
    fold_metrics_df = evaluate_multioutput_model(y_val_fold, y_val_pred)
    fold_metrics_df['Fold'] = fold
    cv_results.append(fold_metrics_df)

# Concatenate all fold results and calculate mean and std deviation
all_cv_metrics = pd.concat(cv_results)
cv_summary = all_cv_metrics.groupby('Target').agg(
    RMSE_mean=('RMSE', 'mean'),
    RMSE_std=('RMSE', 'std'),
    R2_mean=('R2', 'mean'),
    R2_std=('R2', 'std')
).round(3)
print(cv_summary)

           RMSE_mean  RMSE_std  R2_mean  R2_std
Target                                         
DE_heart       1.578     0.162    0.030   0.062
DE_kidney      1.630     0.287    0.026   0.082
DE_liver       1.372     0.097    0.174   0.145
DE_lung        1.486     0.249    0.018   0.113
DE_spleen      1.606     0.239    0.202   0.147
DE_tumor       1.478     0.119    0.028   0.087


## XGBoost

In [ ]:
import xgboost as xgb
from sklearn.model_selection import cross_val_score

# Defining the objective function
def objective(trial):
    params = {
    "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
    "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
    # "max_depth": trial.suggest_int("max_depth", 3, 10),
    # "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
    "gamma": trial.suggest_float("gamma", 0, 5),
    "subsample": trial.suggest_float("subsample", 0.6, 1),
    "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1),
    "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10, log=True),
    "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10, log=True),
    }

    model = xgb.XGBRegressor(**params)

    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    rmse_scores = []

    for tr_idx, va_idx in cv.split(X_train):
        X_tr, X_va = X_train[tr_idx], X_train[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_va)
        rmse_scores.append(rmse(y_va, y_pred))

    return np.mean(rmse_scores)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

print("Best params:", study.best_params)

[I 2026-06-12 13:05:51,727] A new study created in memory with name: no-name-43e87a18-a38e-477c-a2ea-85375e858c45
[I 2026-06-12 13:06:00,374] Trial 0 finished with value: 1.5281371116638183 and parameters: {'n_estimators': 902, 'learning_rate': 0.006233099282439425, 'gamma': 0.8637716246771276, 'subsample': 0.937381244703898, 'colsample_bytree': 0.819112247736117, 'reg_alpha': 5.207927889436867, 'reg_lambda': 0.009812976545888455}. Best is trial 0 with value: 1.5281371116638183.
[I 2026-06-12 13:06:02,841] Trial 1 finished with value: 1.514900803565979 and parameters: {'n_estimators': 340, 'learning_rate': 0.01012340651541907, 'gamma': 2.529591858273059, 'subsample': 0.7462335920137481, 'colsample_bytree': 0.6880603764663903, 'reg_alpha': 1.7162237809954093, 'reg_lambda': 0.1006501662325765}. Best is trial 1 with value: 1.514900803565979.
[I 2026-06-12 13:06:04,592] Trial 2 finished with value: 1.4915699720382691 and parameters: {'n_estimators': 435, 'learning_rate': 0.064741042474601,

Best params: {'n_estimators': 581, 'learning_rate': 0.005023835576468886, 'gamma': 0.03188931636889086, 'subsample': 0.6288262469580669, 'colsample_bytree': 0.8327488393938064, 'reg_alpha': 0.012397408835433166, 'reg_lambda': 0.00033776934482930066}


In [ ]:
#Evaluation for test set
best_params= study.best_params
Best_model=xgb.XGBRegressor(**best_params)

# Fit the best model to the training data
Best_model.fit(X_train, y_train)
#Evaluation for the test data
y_pred_test= Best_model.predict(X_test)
y_pred_test_df = pd.DataFrame(y_pred_test, columns=target_cols, index=y_test.index)
test_metrics = evaluate_multioutput_model(y_test, y_pred_test_df)

print("\n--- Test Set Evaluation Results (per tissue) ---")
print(test_metrics)


--- Test Set Evaluation Results (per tissue) ---
      Target  RMSE    R2
0   DE_tumor  1.81 -0.02
1   DE_heart  1.45  0.25
2   DE_liver  1.41  0.27
3  DE_spleen  1.65  0.08
4    DE_lung  1.39  0.11
5  DE_kidney  1.42  0.03


In [ ]:
from sklearn.model_selection import KFold


# Initialize KFold for cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# List to store evaluation metrics for each fold
cv_results = []

# Loop through each fold
for fold, (train_index, val_index) in enumerate(kf.split(X_train), 1):

    # Split data for the current fold
    X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]
    y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

    # Build and train the model for this fold using the best parameters
    model = xgb.XGBRegressor(**best_params)
    model.fit(X_train_fold, y_train_fold)

    # Make predictions on the validation set
    y_val_pred = model.predict(X_val_fold)

    # Evaluate per-tissue metrics for this fold
    fold_metrics_df = evaluate_multioutput_model(y_val_fold, y_val_pred)
    fold_metrics_df['Fold'] = fold
    cv_results.append(fold_metrics_df)

# Concatenate all fold results and calculate mean and std deviation
all_cv_metrics = pd.concat(cv_results)
cv_summary = all_cv_metrics.groupby('Target').agg(
    RMSE_mean=('RMSE', 'mean'),
    RMSE_std=('RMSE', 'std'),
    R2_mean=('R2', 'mean'),
    R2_std=('R2', 'std')
).round(3)
print(cv_summary)

           RMSE_mean  RMSE_std  R2_mean  R2_std
Target                                         
DE_heart       1.570     0.126    0.010   0.262
DE_kidney      1.518     0.172    0.064   0.482
DE_liver       1.294     0.125    0.260   0.198
DE_lung        1.370     0.139    0.136   0.219
DE_spleen      1.480     0.142    0.310   0.145
DE_tumor       1.434     0.149    0.062   0.255


## LGBM

In [ ]:
import lightgbm
from sklearn.model_selection import cross_val_score
from sklearn.multioutput import MultiOutputRegressor

# Defining the objective function
def objective(trial):
    params = {
    "n_estimators": trial.suggest_int("n_estimators", 100, 2000),
    "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
    "num_leaves": trial.suggest_int("num_leaves", 16, 64),
    "max_depth": trial.suggest_int("max_depth", 3, 12),
    "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
    "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10, log=True),
    "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10, log=True),
    "verbosity":-1,
    "random_state": 42
    }

    model = MultiOutputRegressor(lightgbm.LGBMRegressor(**params))

    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    rmse_scores = []

    for tr_idx, va_idx in cv.split(X_train):
        X_tr, X_va = X_train[tr_idx], X_train[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_va)
        rmse_scores.append(rmse(y_va, y_pred))

    return np.mean(rmse_scores)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

print("Best params:", study.best_params)

[I 2026-06-12 13:11:37,211] A new study created in memory with name: no-name-fc817a7e-c89f-4bf2-af33-66a583fcafce
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LG

Best params: {'n_estimators': 665, 'learning_rate': 0.005527137666404099, 'num_leaves': 57, 'max_depth': 3, 'min_child_samples': 12, 'reg_alpha': 0.5506146709767358, 'reg_lambda': 1.2820724300377864}


In [ ]:
#Evaluation for test set
import lightgbm

best_params= study.best_params
Best_model=MultiOutputRegressor(lightgbm.LGBMRegressor(**best_params))

# Fit the best model to the training data
Best_model.fit(X_train, y_train)
#Evaluation for the test data
y_pred_test= Best_model.predict(X_test)
y_pred_test_df = pd.DataFrame(y_pred_test, columns=target_cols, index=y_test.index)
test_metrics = evaluate_multioutput_model(y_test, y_pred_test_df)

print("\n--- Test Set Evaluation Results (per tissue) ---")
print(test_metrics)


--- Test Set Evaluation Results (per tissue) ---
      Target  RMSE    R2
0   DE_tumor  1.84 -0.06
1   DE_heart  1.65  0.03
2   DE_liver  1.56  0.10
3  DE_spleen  1.70  0.01
4    DE_lung  1.45  0.04
5  DE_kidney  1.54 -0.15


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

In [ ]:
from sklearn.model_selection import KFold


# Initialize KFold for cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# List to store evaluation metrics for each fold
cv_results = []

# Loop through each fold
for fold, (train_index, val_index) in enumerate(kf.split(X_train), 1):

    # Split data for the current fold
    X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]
    y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

    # Build and train the model for this fold using the best parameters
    model = MultiOutputRegressor(lightgbm.LGBMRegressor(**best_params))
    model.fit(X_train_fold, y_train_fold)

    # Make predictions on the validation set
    y_val_pred = model.predict(X_val_fold)

    # Evaluate per-tissue metrics for this fold
    fold_metrics_df = evaluate_multioutput_model(y_val_fold, y_val_pred)
    fold_metrics_df['Fold'] = fold
    cv_results.append(fold_metrics_df)

# Concatenate all fold results and calculate mean and std deviation
all_cv_metrics = pd.concat(cv_results)
cv_summary = all_cv_metrics.groupby('Target').agg(
    RMSE_mean=('RMSE', 'mean'),
    RMSE_std=('RMSE', 'std'),
    R2_mean=('R2', 'mean'),
    R2_std=('R2', 'std')
).round(3)
print(cv_summary)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

           RMSE_mean  RMSE_std  R2_mean  R2_std
Target                                         
DE_heart       1.644     0.130   -0.076   0.247
DE_kidney      1.636     0.139   -0.050   0.414
DE_liver       1.430     0.115    0.100   0.192
DE_lung        1.536     0.198   -0.072   0.237
DE_spleen      1.644     0.197    0.156   0.194
DE_tumor       1.490     0.113    0.000   0.189


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v